# Deckard DVCLive: Native Runtime Walkthrough

This notebook demonstrates deckard-native DVCLive behavior using a real example experiment configuration.

For generic DVC concepts and command semantics, see [DVC Overview](../overview/dvc).

## Optional Dependencies

DVCLive support is optional and requires the `dvclive` extra in your environment.

Install from repository root:

```bash
pip install -e '.[docs]' dvclive
```

If your environment is missing DVCLive, this notebook may fail when running the experiment plugin hooks.

## Goals

This notebook walks through a DVCLive run in four steps:
- generate sklearn default `params.yaml` and `dvc.yaml` under `docs/notebooks/build/dvclive`
- execute the experiment with the DVCLive plugin enabled
- demonstrate component and stage scoping in runtime scores
- demonstrate parameter scoping and system monitoring outputs

In [1]:
from __future__ import annotations

import json
import shutil
from pathlib import Path

from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from hydra.utils import instantiate
from omegaconf import OmegaConf

from deckard.experiment import build_dvc_stage_plan, generate_dvc_pipeline
from deckard.experiment.base import ExperimentConfig


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'deckard').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root containing pyproject.toml')


def build_default_example_experiment(repo_root: Path):
    config_dir = repo_root / 'examples/sklearn/config'
    GlobalHydra.instance().clear()
    with initialize_config_dir(version_base=None, config_dir=config_dir.as_posix()):
        cfg = compose(config_name='default')

    payload = OmegaConf.to_container(cfg, resolve=True)
    allowed_keys = set(ExperimentConfig.__dataclass_fields__.keys()) | {'_target_'}
    filtered = {key: value for key, value in payload.items() if key in allowed_keys}
    filtered.setdefault('_target_', 'deckard.ExperimentConfig')

    experiment = instantiate(filtered)
    experiment.experiment_name = f"{getattr(experiment, 'experiment_name', 'default')}-dvclive"
    return cfg, experiment


REPO_ROOT = find_repo_root(Path.cwd())
DVCLIVE_DIR = REPO_ROOT / 'docs/notebooks/build/dvclive'

if DVCLIVE_DIR.exists():
    shutil.rmtree(DVCLIVE_DIR)
DVCLIVE_DIR.mkdir(parents=True, exist_ok=True)

cfg, experiment = build_default_example_experiment(REPO_ROOT)

generated_params_file = DVCLIVE_DIR / 'params.yaml'
generated_dvc_file = DVCLIVE_DIR / 'dvc.yaml'

plugin_cfg = {
    'enabled': True,
    'dvclive_dir': DVCLIVE_DIR.as_posix(),
    'mode': 'single',
    'pull_dependencies': False,
    'push_outputs': False,
    'make_summary': True,
    'make_report': True,
    'make_dvcyaml': True,
    'report_mode': 'html',
    'resume': False,
    'save_dvc_exp': False,
    'cache_images': False,
    'monitor_system': True,
    'fail_on_dvc_error': False,
    'dvc_file': generated_dvc_file.as_posix(),
    'params_file': generated_params_file.as_posix(),
}

experiment.dvc_plugin = plugin_cfg

files_cfg = getattr(experiment, 'files', None)
runtime_alias_updates = {}
if files_cfg is not None and hasattr(files_cfg, 'as_dict') and hasattr(files_cfg, 'update'):
    if not getattr(files_cfg, 'params_file', None):
        runtime_alias_updates['params_file'] = generated_params_file.as_posix()
    if not getattr(files_cfg, 'score_file', None):
        runtime_alias_updates['score_file'] = (DVCLIVE_DIR / 'scores.json').as_posix()
    if not getattr(files_cfg, 'log_file', None):
        runtime_alias_updates['log_file'] = (DVCLIVE_DIR / 'deckard.log').as_posix()
    if not getattr(files_cfg, 'error_file', None):
        runtime_alias_updates['error_file'] = (DVCLIVE_DIR / 'deckard.err').as_posix()
    if runtime_alias_updates:
        files_cfg.update(**runtime_alias_updates)

stage_plan = build_dvc_stage_plan(
    experiment,
    mode='single',
    params_file=generated_params_file.as_posix(),
    dvc_file=generated_dvc_file.as_posix(),
)
pipeline_payload = generate_dvc_pipeline(
    experiment,
    output_file=generated_dvc_file.as_posix(),
    params_file=generated_params_file.as_posix(),
    mode='single',
    overwrite=True,
)

print('prepared build directory:', DVCLIVE_DIR)
print('generated params:', generated_params_file.relative_to(REPO_ROOT))
print('generated dvc:', generated_dvc_file.relative_to(REPO_ROOT))
print('native dvc stage count:', len(stage_plan))
print('native dvc stage names:', [entry['name'] for entry in stage_plan][:10])

/Users/c.meyers/.pyenv/versions/3.10.20/envs/deckard/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


prepared build directory: /Users/c.meyers/Documents/deckard/docs/notebooks/build/dvclive
generated params: docs/notebooks/build/dvclive/params.yaml
generated dvc: docs/notebooks/build/dvclive/dvc.yaml
native dvc stage count: 15
native dvc stage names: ['data__load', 'data__sample', 'data__pipeline', 'data__data-score', 'data__data-persist', 'defense__apply-fit-defense', 'model__train', 'defense__apply-predict-defense-defense1', 'model__model-score', 'model__model-persist']


## Execute Experiment

Now run the configured sklearn default experiment using the DVCLive plugin and collect runtime scores for scoping analysis.

In [2]:
scores = experiment()
score_payload = dict(scores) if isinstance(scores, dict) else {}

summary_path = DVCLIVE_DIR / 'dvclive' / 'summary.json'
generated_params_file = DVCLIVE_DIR / 'params.yaml'
generated_dvc_file = DVCLIVE_DIR / 'dvc.yaml'

print('score payload type:', type(scores).__name__)
print('score payload size:', len(score_payload))
print('summary exists:', summary_path.exists())
print('params exists:', generated_params_file.exists())
print('dvc exists:', generated_dvc_file.exists())

score payload type: dict
score payload size: 54
summary exists: False
params exists: True
dvc exists: True


## Component and Stage Scoping

Deckard score keys are namespaced by component and execution stage.

The next cell groups score keys by component prefix and highlights stage-scoped keys.

In [3]:
score_payload = dict(scores) if isinstance(scores, dict) else {}
score_keys = sorted(score_payload.keys())

component_buckets = {}
for key in score_keys:
    if '__' in key:
        component = key.split('__', 1)[0]
    elif '/' in key:
        component = key.split('/', 1)[0]
    else:
        component = key.split('_', 1)[0]
    component_buckets.setdefault(component, []).append(key)

stage_keys = [
    key
    for key in score_keys
    if any(token in key.lower() for token in ('stage', '__', 'attack', 'model', 'data', 'score'))
]

print('score key count:', len(score_keys))
print('component bucket count:', len(component_buckets))
print('--- component scoping (sample) ---')
for component in sorted(component_buckets)[:15]:
    print(f'- {component}: {len(component_buckets[component])} keys')

print('--- stage-scoped/runtime keys (sample) ---')
for key in stage_keys[:40]:
    print('-', key)

score key count: 54
component bucket count: 18
--- component scoping (sample) ---
- accuracy: 1 keys
- attack: 6 keys
- benign: 7 keys
- data: 4 keys
- defense: 1 keys
- evasion: 5 keys
- f1: 1 keys
- files: 1 keys
- log: 1 keys
- pipeline: 4 keys
- precision: 1 keys
- prediction: 3 keys
- recall: 1 keys
- roc: 1 keys
- test: 2 keys
--- stage-scoped/runtime keys (sample) ---
- attack_execution_order
- attack_generation_time
- attack_prediction_time
- attack_score_time
- attack_size
- attack_stage
- benign_attack_score_time
- benign_attack_size
- data_load_time
- data_pipeline_time
- data_sample_time
- data_score_time
- evasion_f1-score
- prediction_score_time
- untargeted_evasion_benign_attack_score_time
- untargeted_evasion_benign_attack_size
- untargeted_evasion_f1-score


## Parameter Scoping

This section inspects generated params and summarizes how keys scope across components and stage-relevant settings.

In [4]:
summary_path = DVCLIVE_DIR / 'dvclive' / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('summary path:', summary_path.relative_to(REPO_ROOT))
    print('summary keys:', sorted(summary.keys())[:20])
else:
    summary = {}
    print('summary.json not found at', summary_path)

params_file = DVCLIVE_DIR / 'params.yaml'
print('params exists:', params_file.exists())

def flatten_items(payload, prefix=''):
    if isinstance(payload, dict):
        for key, value in payload.items():
            child_prefix = f"{prefix}.{key}" if prefix else str(key)
            yield from flatten_items(value, child_prefix)
    elif isinstance(payload, list):
        for idx, value in enumerate(payload):
            child_prefix = f"{prefix}[{idx}]"
            yield from flatten_items(value, child_prefix)
    else:
        yield prefix, payload

params_payload = {}
if params_file.exists():
    params_payload = OmegaConf.to_container(OmegaConf.load(params_file), resolve=True)
elif hasattr(experiment, 'to_dict') and callable(experiment.to_dict):
    params_payload = experiment.to_dict()
    print('using experiment config fallback for param inspection')

flat_params = list(flatten_items(params_payload))
print('flattened param entries:', len(flat_params))

scope_counts = {}
for key, _ in flat_params:
    head = key.split('.', 1)[0] if key else 'root'
    scope_counts[head] = scope_counts.get(head, 0) + 1

print('--- parameter scope counts ---')
for head in sorted(scope_counts, key=scope_counts.get, reverse=True)[:15]:
    print(f'- {head}: {scope_counts[head]} entries')

stage_param_entries = [
    (key, value)
    for key, value in flat_params
    if any(token in key.lower() for token in ('stage', 'mode', 'dvc_plugin', 'attack', 'model'))
]

print('--- stage/plugin-relevant params (sample) ---')
for key, value in stage_param_entries[:30]:
    print(f'- {key}: {value!r}')

print('stage/plugin relevant param count:', len(stage_param_entries))

summary.json not found at /Users/c.meyers/Documents/deckard/docs/notebooks/build/dvclive/dvclive/summary.json
params exists: True
flattened param entries: 753
--- parameter scope counts ---
- experiment: 593 entries
- runtime: 136 entries
- params: 23 entries
- schema_version: 1 entries
--- stage/plugin-relevant params (sample) ---
- experiment.default_stage: 'post-pipeline'
- experiment.data.default_stage: 'post-pipeline'
- experiment.data.score_mode: 'test'
- experiment.data.score_stage: 'post-pipeline'
- experiment.model.id: ''
- experiment.model.path: ''
- experiment.model.payload_kind: 'data'
- experiment.model.default_stage: 'post-pipeline'
- experiment.model.name: 'sklearn.ensemble.RandomForestClassifier'
- experiment.model.classifier: True
- experiment.model.model_params.bootstrap: True
- experiment.model.model_params.ccp_alpha: 0.0
- experiment.model.model_params.class_weight: None
- experiment.model.model_params.criterion: 'gini'
- experiment.model.model_params.max_depth: Non

## DVC Outputs

The plugin writes runtime artifacts used by DVC and docs workflows. The cell below enumerates generated files under the DVCLive output root.

In [5]:
for path in sorted(DVCLIVE_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(REPO_ROOT))

docs/notebooks/build/dvclive/dvc.yaml
docs/notebooks/build/dvclive/params.runtime_cache.pkl
docs/notebooks/build/dvclive/params.yaml
docs/notebooks/build/dvclive/scores.json


## System Monitoring Features

With `monitor_system=True`, Deckard/DVCLive can capture CPU, memory, and disk telemetry and attach it to runtime outputs. The next cell prints any discovered system monitoring scores and related generated files.

In [6]:
print('monitor_system configured:', bool(plugin_cfg.get('monitor_system', False)))

monitor_scores = {}
if isinstance(score_payload, dict):
    monitor_scores.update({
        key: value
        for key, value in score_payload.items()
        if str(key).startswith('system_monitor/')
    })

outputs_payload = getattr(experiment, 'outputs', {})
if isinstance(outputs_payload, dict):
    dvclive_payload = outputs_payload.get('dvclive', {})
    if isinstance(dvclive_payload, dict):
        nested_monitor = dvclive_payload.get('system_monitor_scores', {})
        if isinstance(nested_monitor, dict):
            monitor_scores.update(nested_monitor)

print('system monitor metric count:', len(monitor_scores))
for key in sorted(monitor_scores):
    print(f'- {key}: {monitor_scores[key]}')

print('--- system-monitor related files ---')
system_files = [
    path.relative_to(REPO_ROOT)
    for path in DVCLIVE_DIR.rglob('*')
    if path.is_file() and any(token in path.name.lower() for token in ('system', 'cpu', 'memory', 'disk'))
]
if system_files:
    for path in sorted(system_files):
        print(path)
else:
    print('No system-monitor file names found in dvclive outputs.')
    print('Checked score payload + experiment outputs + dvclive artifact filenames.')

monitor_system configured: True
system monitor metric count: 0
--- system-monitor related files ---
No system-monitor file names found in dvclive outputs.
Checked score payload + experiment outputs + dvclive artifact filenames.
